### 📊 Mục tiêu của notebook

Notebook minh họa **toàn bộ lý thuyết Phần 2** từ dữ liệu AmesHousing lập có kiểm soát, bao gồm:
1. **Khảo sát Dữ Liệu (EDA)** — Thống kê mô tả, kiểm tra tính toàn vẹn và trực quan hóa phân phối của biến mục tiêu (SalePrice) cùng các biến dự báo.
2. **Tiền xử lý & Feature Engineering** — Xử lý các giá trị khuyết thiếu (missing values) và thiết kế thêm các đặc trưng mới để tối ưu sức mạnh mô hình.
3. **Data Pipeline & VIF** — Xây dựng luồng xử lý dữ liệu tự động (Encoding, Scaling, Handling Outliers) và kiểm tra hiện tượng đa cộng tuyến (VIF).
4. **Đánh Giá Mô Hình** — Huấn luyện, so sánh và đo lường hiệu suất thực tế của các mô hình trên tập Test set.

---

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
import matplotlib.ticker as mticker
from scipy import stats as scipy_stats
from scipy.stats import norm, skew, kurtosis
from matplotlib.patches import Patch
from sklearn.model_selection import train_test_split
from sklearn.metrics import pairwise_distances
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# ── Điều chỉnh sys.path an toàn và linh hoạt ────────────────────────
CURRENT_DIR = os.getcwd()

# Kiểm tra xem thư mục hiện tại đang là gốc (Project root) hay là thư mục con 'part2'
if os.path.basename(CURRENT_DIR) == 'part2':
    BASE_DIR = os.path.dirname(CURRENT_DIR)
    PART2_DIR = CURRENT_DIR
else:
    BASE_DIR = CURRENT_DIR
    PART2_DIR = os.path.join(BASE_DIR, 'part2')

# Đưa vào sys.path (chỉ thêm những đường dẫn cần thiết, loại bỏ dư thừa)
for p in [BASE_DIR, PART2_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Đường dẫn dữ liệu ────────────────────────────────────────────────
# Tận dụng lại biến PART2_DIR thay vì gọi lại BASE_DIR và string 'part2'
DATA_PATH = os.path.join(PART2_DIR, 'data', 'AmesHousing.csv')

from part2.model_comparison import PolynomialFeatureGenerator, InteractionFeatureGenerator
from part2.clean_data import clean_data
from part2.data_pipeline import DataPipeline, run_vif_check
from part2.model_comparison import (
    OLSBasic, OLSFeatureSelector, RidgeCV, LassoCV,
    PolynomialFeatureGenerator, InteractionFeatureGenerator,
    train_test_split_df, evaluate_model,
)
from part2.advanced_methods import KernelRidgeRegressor, BayesianLinearRegressor


print('Python:', sys.version.split()[0])
print('Numpy :', np.__version__)
print('Pandas:', pd.__version__)
print('Data  :', DATA_PATH)

 # Phần 1: Khảo Sát Dữ Liệu (Exploratory Data Analysis — EDA)

## 1.1. Kiểm tra chất lượng dữ liệu cơ bản

### 1.1.1. Thống kê mô tả

In [ ]:
# Cài đặt style cho đồ thị
sns.set_theme(style="whitegrid")

# Load dữ liệu sử dụng biến DATA_PATH từ cell import
df = pd.read_csv(DATA_PATH)

# Loại bỏ khoảng trắng thừa trong tên cột
df.columns = df.columns.str.strip()

print(f"Kích thước bộ dữ liệu: {df.shape}")

# Thống kê mô tả (mean, median, std, min, max, quartiles)
display(df.describe())

# Xem qua các biến phân loại
display(df.describe(exclude='number'))



### 1.1.2. Kiểm tra dữ liệu trùng lắp

In [ ]:
num_duplicates = df.duplicated().sum()
print(f"Số lượng dòng dữ liệu bị trùng lắp (Duplicates): {num_duplicates}")

#### Nhận xét: Kiểm tra dữ liệu trùng lắp
Qua hàm `duplicated()`, ta thấy bộ dữ liệu Ames Housing không có dòng dữ liệu nào bị trùng lắp hoàn toàn (0 duplicates). Điều này chứng tỏ khâu thu thập dữ liệu gốc đã được quản lý định danh (thông qua mã PID của từng căn nhà) rất tốt. Do đó, nhóm không cần thực hiện bước xóa dòng trùng (drop duplicates) trong quá trình tiền xử lý.

## 1.2. Trực quan hóa dữ liệu (Histogram & Boxplot)

### 1.2.1. Khảo sát Biến mục tiêu (SalePrice)

In [ ]:
# Thiết kế biểu đồ phân tích cho biến mục tiêu
plt.figure(figsize=(12, 6))

# Vẽ Histogram kết hợp đường KDE
sns.histplot(df['SalePrice'], kde=True, stat="density", color="salmon", bins=50)

# Chèn thêm đường cong phân phối chuẩn (Normal Distribution) để so sánh độ lệch

mu, std = scipy_stats.norm.fit(df['SalePrice'].dropna())
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = scipy_stats.norm.pdf(x, mu, std)
plt.plot(x, p, 'k', linewidth=2, linestyle='--', label=rf'Normal Dist. ($\mu$={mu:.0f}, $\sigma$={std:.0f})')

plt.title('Phân phối của biến mục tiêu: SalePrice', fontsize=16)
plt.xlabel('Giá bán (USD)', fontsize=12)
plt.ylabel('Mật độ (Density)', fontsize=12)
plt.legend()
plt.show()

# Vẽ Boxplot cho SalePrice để soi outliers của y
plt.figure(figsize=(10, 2))
sns.boxplot(x=df['SalePrice'], color='salmon')
plt.title('Boxplot của SalePrice', fontsize=14)
plt.show()
# Định lượng mức độ lệch
print(f"Độ lệch (Skewness) của SalePrice: {df['SalePrice'].skew():.2f}")

#### Nhận xét về Biến mục tiêu (Target Variable Analysis)
* **Phân phối:** Biểu đồ Histogram cho thấy `SalePrice` bị lệch phải (Right-skewed) rõ rệt. Đỉnh của phân phối tập trung ở khoảng 130,000 - 160,000 USD, nhưng có một dải đuôi dài kéo về phía các mức giá cao (trên 500,000 USD).
* **So sánh với Phân phối chuẩn:** Đường cong thực tế (Salmon) lệch đáng kể so với đường lý thuyết (Black dashed), cho thấy dữ liệu không tuân theo phân phối chuẩn.
* **Outliers:** Boxplot xác nhận sự tồn tại của nhiều giá trị ngoại lệ ở phía cận trên (giá cao).
* **Kết luận cho OLS:** Để thỏa mãn giả định về phân phối chuẩn của phần dư trong mô hình OLS, việc áp dụng phép biến đổi **Log-transformation** cho `SalePrice` là bắt buộc trước khi đưa vào huấn luyện.

### 1.2.2. Khảo sát Biến dự báo (Predictors)

In [ ]:
# Tìm các biến có tương quan cao nhất với SalePrice để ưu tiên khảo sát
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

# Lấy top 30 biến tương quan mạnh nhất với SalePrice
top_30_features = corr_matrix['SalePrice'].abs().sort_values(ascending=False).head(30).index

# Định nghĩa danh sách các biến mang tính chất thời gian (năm/tháng)
temporal_cols = ['Year Built', 'Year Remod/Add', 'Garage Yr Blt', 'Yr Sold', 'Mo Sold']

# Lọc ra top các biến tương quan cao, bỏ qua biến mục tiêu (index 0)
# và bỏ qua các biến nằm trong danh sách temporal_cols
predictors_to_plot = [col for col in top_30_features[1:] if col not in temporal_cols][:12]

print(f"12 biến được chọn để vẽ phân phối: {predictors_to_plot}")

# Vẽ lưới Histogram 3x4
plt.figure(figsize=(20, 15))
for i, col in enumerate(predictors_to_plot, 1):
    plt.subplot(3, 4, i)
    n_unique = df[col].dropna().nunique()
    if n_unique <= 15:
        sns.countplot(data=df, x=col, color='skyblue')
    else:
        sns.histplot(df[col], kde=True, bins=30, color='skyblue')
    plt.title(f'Phân phối của {col}', fontsize=14)
    plt.xlabel(col, fontsize=12, fontweight='bold')
    plt.ylabel('Tần suất', fontsize=12)
plt.tight_layout()
plt.show()

# Vẽ lưới Boxplot 3x4
plt.figure(figsize=(20, 15))
for i, col in enumerate(predictors_to_plot, 1):
    plt.subplot(3, 4, i)
    sns.boxplot(y=df[col], color='lightgreen')
    plt.title(f'Boxplot của {col}', fontsize=14)
    plt.ylabel('Giá trị', fontsize=12)
    plt.xlabel(col, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

#### Nhận xét về Phân phối của các Biến dự báo
Qua quan sát biểu đồ Histogram của 12 biến độc lập quan trọng nhất, ta rút ra các đặc điểm sau:

* **Các biến đo lường diện tích (`Gr Liv Area`, `Total Bsmt SF`, `1st Flr SF`):** Hầu hết đều có xu hướng lệch phải (Right-skewed). Điều này phản ánh thực tế cấu trúc nhà ở: đại đa số là nhà có diện tích trung bình, chỉ một thiểu số ít là các dinh thự hoặc nhà có diện tích cực lớn.
* **Các biến đánh giá chất lượng (`Overall Qual`, `Garage Cars`, `Full Bath`):** Có dạng phân phối đa đỉnh (Multimodal) hoặc phân bố theo cụm rời rạc. Đây là đặc trưng hiển nhiên của các biến mang tính chất thứ bậc (Ordinal) hoặc số đếm (Count).
* **Đề xuất xử lý:** Sự lệch phải của các biến diện tích có thể làm tăng phương sai của phần dư trong mô hình OLS. Nhóm sẽ cân nhắc áp dụng phép biến đổi Logarit (Log-transform) cho các biến số học liên tục bị lệch nặng để đưa chúng về dạng chuẩn hơn trong bước Pipeline.

#### Nhận xét về Giá trị ngoại lệ (Visual Outliers) của các Biến dự báo
Qua quan sát hình dáng biểu đồ Boxplot, ta nhận thấy sự xuất hiện của rất nhiều điểm ngoại lệ (các chấm đen nằm ngoài râu của đồ thị):

* **Các điểm đòn bẩy tiềm năng (Potential Leverage Points):** Dễ thấy nhất ở biến `Gr Liv Area` và `Total Bsmt SF`. Có những quan sát nằm tách biệt hoàn toàn ở phía trên đỉnh đồ thị (đại diện cho các căn nhà có diện tích cực lớn). Trong lý thuyết OLS, đây là những điểm có nguy cơ trở thành "đòn bẩy", kéo lệch đường hồi quy của toàn bộ tập dữ liệu.
* **Các biến tiện ích đặc thù:** Những biến như `Mas Vnr Area` (Diện tích ốp gạch/đá) hay `Wood Deck SF` (Diện tích sàn gỗ) có mật độ điểm ngoại lệ rất dày đặc. Lý do trực quan là phần lớn các căn nhà không có tiện ích này (giá trị = 0), khiến phần thân hộp (hộp chứa 50% dữ liệu) bị bóp nghẹt lại, đẩy các giá trị > 0 ra ngoài khoảng râu đồ thị.
* **Định hướng bước tiếp theo:** Từ những quan sát trực quan này, nhóm sẽ tiến hành định lượng cụ thể số lượng ngoại lệ bằng công thức thống kê ở phần "Phân tích chuyên sâu" bên dưới để có cơ sở đưa ra thuật toán xử lý (Winsorization) trong `DataPipeline`.

### 1.2.3. Đánh giá mức độ tương quan (Heatmap)

In [ ]:
# Lấy top 15 biến có độ tương quan cao nhất với SalePrice
top_15_features = corr_matrix['SalePrice'].abs().sort_values(ascending=False).head(15).index
top_15_corr = df[top_15_features].corr()

# Vẽ Clustermap để thuật toán tự động gom các biến đa cộng tuyến lại gần nhau
cluster = sns.clustermap(top_15_corr,
                         annot=True,
                         fmt=".2f",
                         cmap='coolwarm',
                         linewidths=0.5,
                         figsize=(12, 10), # Clustermap dùng figsize bên trong hàm
                         annot_kws={"size": 10},
                         tree_kws={"linewidths": 1.5}) # Làm nét đường cây phân cụm

cluster.fig.suptitle('Ma trận tương quan phân cụm (Clustermap) của 15 biến quan trọng', fontsize=16, y=1.05)

plt.setp(cluster.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
plt.setp(cluster.ax_heatmap.get_yticklabels(), rotation=0)

plt.show()

#### Nhận xét về Ma trận tương quan phân cụm (Clustermap)
Thay vì sắp xếp thủ công, nhóm sử dụng thuật toán Phân cụm phân cấp (Hierarchical Clustering) thông qua biểu đồ Clustermap để máy tính **tự động** tính toán khoảng cách và gom nhóm các biến có độ tương quan cao lại với nhau. Qua quan sát, ta rút ra các kết luận quan trọng:

* **Các biến dự báo "át chủ bài":** `Overall Qual` (Chất lượng tổng thể) và `Gr Liv Area` (Diện tích sinh hoạt) đã khẳng định vị thế là 2 biến độc lập có tác động mạnh mẽ nhất đến `SalePrice`. Đây sẽ là các biến mang trọng số cao trong phương trình hồi quy.
* **Phát hiện Đa cộng tuyến (Multicollinearity) tự động:** Nhờ cây phân cụm (dendrogram), thuật toán đã nhóm thành công 4 cụm biến (các ô màu đỏ sẫm tạo thành khối vuông) có hiện tượng đa cộng tuyến rất cao:
  1. **`Garage Cars` & `Garage Area`:** Sức chứa (số xe) và diện tích vật lý của gara tỉ lệ thuận trực tiếp với nhau.
  2. **`TotRms AbvGrd` & `Gr Liv Area`:** Tổng số phòng và tổng diện tích sinh hoạt trên mặt đất về cơ bản đang đo lường cùng một quy mô không gian.
  3. **`1st Flr SF` & `Total Bsmt SF`:** Khuôn viên diện tích mặt bằng tầng 1 thường được xây dựng khớp với diện tích tầng hầm bên dưới nó.
  4. **`Year Built` & `Garage Yr Blt`:** Đại đa số gara đều được xây dựng cùng năm với thời điểm cất nhà chính.
* **Đề xuất xử lý cho mô hình OLS:** Hiện tượng đa cộng tuyến sẽ làm ma trận thiết kế bị suy biến và gây nhiễu các hệ số $\beta$. Ở bước `DataPipeline`, nhóm đề xuất sẽ **chỉ giữ lại một biến** trong mỗi cặp để đưa vào huấn luyện. Điển hình như cặp số 4, nhóm sẽ loại bỏ `Garage Yr Blt` (vì cột này chứa 159 giá trị khuyết) và giữ lại `Year Built` nhằm tối ưu hóa bộ dữ liệu.

#### 1.2.4. Khảo sát tính tuyến tính giữa các Biến độc lập và Biến mục tiêu (SalePrice)

In [ ]:
plt.figure(figsize=(18, 6))

# 1. Gr Liv Area vs SalePrice (Biến diện tích liên tục)
plt.subplot(1, 2, 1)
sns.regplot(data=df, x='Gr Liv Area', y='SalePrice',
            scatter_kws={'alpha':0.5, 's':20},
            line_kws={'color':'red'})
plt.title('Mối quan hệ giữa Diện tích sinh hoạt và Giá bán', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)

# 2. Overall Qual vs SalePrice (Biến chất lượng rời rạc)
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='Overall Qual', y='SalePrice', hue='Overall Qual', palette='viridis', legend=False)
plt.title('Mối quan hệ giữa Chất lượng tổng thể và Giá bán', fontsize=14)

plt.tight_layout()
plt.show()

#### Nhận xét về tính tuyến tính và tương quan
Biểu đồ Scatter Plot và Boxplot kết hợp với đường hồi quy (Red line) cho thấy:

* **Tính tuyến tính:** Biến `Gr Liv Area` thể hiện một mối quan hệ tuyến tính thuận rất rõ rệt với `SalePrice`. Khi diện tích tăng, giá nhà có xu hướng tăng theo một tỉ lệ ổn định. Điều này cho thấy mô hình hồi quy tuyến tính là một lựa chọn hợp lý để thử nghiệm ở bước tiếp theo.
* **Hiện tượng "loe phễu":** Ở biểu đồ bên trái, ta thấy khi diện tích càng lớn, các điểm dữ liệu càng phân tán rộng hơn (hình cái loa). Đây là dấu hiệu của phương sai thay đổi (Heteroscedasticity), củng cố thêm lý do tại sao nhóm cần áp dụng Log-transform ở bước sau để "nén" phương sai này lại.
* **Biến chất lượng:** Với `Overall Qual`, giá nhà tăng theo bậc thang khi chất lượng tăng. Tuy nhiên, ở các mức chất lượng cao nhất (9 và 10), sự biến động về giá là cực lớn, cho thấy chất lượng chỉ là điều kiện cần, còn giá cuối cùng phụ thuộc thêm nhiều yếu tố khác.

## 1.3. Phân tích chuyên sâu

### 1.3.1. Phân tích Missing Values

In [ ]:
# Thống kê số lượng và tỷ lệ missing
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_report = pd.DataFrame({'Số lượng': missing, 'Tỷ lệ %': missing_pct})
missing_report = missing_report[missing_report['Số lượng'] > 0].sort_values('Tỷ lệ %', ascending=False)
print(f"Có {len(missing_report)} biến bị missing:\n")
display(missing_report)

# Chỉ vẽ các cột có missing
missing_cols = df.columns[df.isnull().any()].tolist()
if missing_cols:
    msno.bar(df[missing_cols], figsize=(12, 5), fontsize=10)
    plt.title("Tỷ lệ missing values")
    plt.show()
else:
    print("Không có cột nào bị missing.")

### Phân tích tổng quan dữ liệu khuyết (Missing Values)

Thay vì liệt kê đơn lẻ, việc phân tích dữ liệu khuyết được nhóm chia thành các cụm đặc trưng có cùng bản chất nghiệp vụ để dễ dàng nhận diện pattern (quy luật bị khuyết). Trực quan hóa dữ liệu cho thấy các giá trị `NaN` không phân bố ngẫu nhiên mà tập trung vào 3 nhóm chính:

1. **Nhóm khuyết diện rộng (> 80%):** Bao gồm `Pool QC` (99.5%), `Misc Feature` (96.3%), `Alley` (93.2%), `Fence` (80.4%). Đây là các tiện ích "xa xỉ" hoặc không phổ biến. Tỉ lệ khuyết cao phản ánh thực tế phần lớn các ngôi nhà không sở hữu các tiện ích này.
2. **Nhóm khuyết theo cụm cấu trúc (Structural Clusters - ~2.5% đến 5%):**
   - **Cụm Garage:** Các biến `Garage Type`, `Garage Finish`, `Garage Qual`, `Garage Cond`, `Garage Yr Blt` luôn bị khuyết cùng lúc trên cùng các dòng dữ liệu.
   - **Cụm Basement (Tầng hầm):** Tương tự, `Bsmt Qual`, `Bsmt Cond`, `Bsmt Exposure`, `BsmtFin Type 1/2` có pattern khuyết đồng thời.
   $\rightarrow$ Trạng thái khuyết này có tính hệ thống cao, đại diện cho việc ngôi nhà hoàn toàn không có Gara hoặc Tầng hầm.
3. **Nhóm khuyết rời rạc (MAR / Lỗi nhập liệu):** - `Lot Frontage` (~16.7%): Mọi lô đất đều phải có mặt tiền tiếp giáp, việc khuyết dữ liệu có thể do khó khăn trong việc đo đạc ở một số khu vực nhất định.
   - `Mas Vnr Type / Area` và `Electrical`: Tỉ lệ khuyết rất nhỏ, mang dấu vết của sự bỏ sót ngẫu nhiên trong quá trình số hóa dữ liệu.

*Nhận định:* Phần lớn dữ liệu khuyết trong tập Ames Housing chứa đựng **thông tin nghiệp vụ quan trọng** (nhà không có tiện ích) thay vì lỗi thu thập. Do đó, tuyệt đối không được dùng phương pháp xóa dòng (Listwise Deletion) để tránh mất mát thông tin nghiêm trọng.

### 1.3.2. Phát hiện Outliers bằng phương pháp IQR

In [ ]:
# 1. ĐỊNH LƯỢNG OUTLIERS BẰNG PHƯƠNG PHÁP IQR
outlier_counts = {}
iqr_zero_cols = []

def count_outliers_iqr_advanced(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1

    # Bắt lỗi: Nếu IQR = 0, phương pháp này trở nên vô nghĩa
    if IQR == 0:
        return -1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = dataframe[(dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)]
    return len(outliers)

# Quét qua toàn bộ các cột numeric
for col in numeric_cols:
    count = count_outliers_iqr_advanced(df, col)
    if count == -1:
        iqr_zero_cols.append(col)
    else:
        outlier_counts[col] = count

# Hiển thị bảng Outliers hợp lệ
outliers_df = pd.DataFrame(list(outlier_counts.items()), columns=['Biến (Feature)', 'Số lượng Outliers'])
outliers_df['Tỉ lệ (%)'] = (outliers_df['Số lượng Outliers'] / len(df)) * 100
outliers_df = outliers_df[outliers_df['Số lượng Outliers'] > 0].sort_values(by='Số lượng Outliers', ascending=False)

print("BẢNG THỐNG KÊ OUTLIERS HỢP LỆ (IQR > 0)")
display(outliers_df.head(10))

print("\nDANH SÁCH CÁC BIẾN KHÔNG THỂ ÁP DỤNG IQR (IQR = 0)")
print(f"Tổng cộng có {len(iqr_zero_cols)} biến. Danh sách chi tiết:")
print(iqr_zero_cols)

# 2. XÁC ĐỊNH ĐIỂM ĐÒN BẨY CỰC ĐOAN
leverage_candidates = df[(df['Gr Liv Area'] > 4000) & (df['SalePrice'] < 300000)]
print(f"\nĐIỂM ĐÒN BẨY TIỀM NĂNG (Gr Liv Area > 4000 & SalePrice < 300k): {len(leverage_candidates)} quan sát")

#### Nhận xét về Giá trị ngoại lệ & Giới hạn của phương pháp IQR
Trong quá trình định lượng Outliers, nhóm phát hiện một vấn đề mang tính nền tảng của thuật toán IQR: Có nhiều biến sở hữu khoảng tứ phân vị bằng 0 ($IQR = 0$). Nhóm đã tiến hành lọc riêng các biến này ra khỏi bảng thống kê và phân loại thành 2 nhóm bản chất:

1. **Nhóm dữ liệu thưa thớt (Zero-inflated):** Tiêu biểu là `BsmtFin SF 2`, `Enclosed Porch`, `Screen Porch`. Hơn 80% giá trị của các biến này là 0 (nhà không có tiện ích này). Khi $Q1 = 0$ và $Q3 = 0$, mọi giá trị $> 0$ đều bị IQR kết luận là ngoại lệ. Điều này là **vô lý về mặt nghiệp vụ**, vì việc có một cái hiên nhà không thể bị coi là điểm dị thường.
2. **Nhóm dữ liệu thiếu phương sai (Near-constant):** Tiêu biểu là `Kitchen AbvGr`. 99% các căn nhà có đúng 1 nhà bếp ($Q1 = 1, Q3 = 1$). Những căn có 2 bếp sẽ bị IQR bắt lỗi. Thực chất đây là những căn hộ thiết kế đặc biệt (nhà ghép/duplex), việc áp dụng công thức khoảng cách cho biến gần như hằng số này là **không phù hợp**.

**Định hướng xử lý trong Pipeline:**
* **Đối với các biến có IQR bình thường (như `SalePrice`, `Gr Liv Area`):** Nhóm sẽ áp dụng kỹ thuật **Winsorization** (chặn phân vị 1% - 99%) để giảm sát thương của các điểm đòn bẩy.
* **Đối với nhóm $IQR = 0$:** Nhóm hoàn toàn **không áp dụng kỹ thuật gọt Outliers**. Thay vào đó, nếu cần thiết, nhóm sẽ xem xét chuyển đổi chúng thành biến phân loại nhị phân (ví dụ: `Has_Enclosed_Porch`: 1/0) để đưa vào mô hình hiệu quả hơn.

#### Nhận xét về Điểm đòn bẩy
Các điểm có Gr Liv Area > 4000 nhưng SalePrice < 300k là bất thường so với xu hướng chung — diện tích lớn nhưng giá lại thấp. Trong OLS, những điểm này có thể trở thành leverage points làm lệch đường hồi quy. Sẽ được loại khỏi tập train trong DataPipeline.

# Phần 2: Tiền xử lý missing values + Feature Engineering

Không phải mọi giá trị thiếu đều có cùng bản chất. Trước khi chọn kỹ thuật xử lý, cần xác định **cơ chế khuyết** (missing mechanism) của từng nhóm biến, vì chọn sai kỹ thuật sẽ tạo ra dữ liệu mâu thuẫn logic và làm sai lệch mô hình.

---

#### Nhóm 1 — Biến định danh và near-zero variance (DROP)

`Order` và `PID` là số thứ tự và mã định danh, unique mỗi hàng — không mang thông tin dự báo, cần drop trước mọi bước xử lý.

`Utilities` (99.9% = `AllPub`) và `Street` (99.6% = `Pave`) gần như hằng số — không phân biệt được các ngôi nhà với nhau, đưa vào mô hình chỉ tạo nhiễu.

**Kỹ thuật: Drop hoàn toàn.**

---

#### Nhóm 2 — Biến >80% missing: MNAR có cấu trúc (DROP + tạo cờ)

`Pool QC` (99.6%), `Misc Feature` (96.4%), `Alley` (93.2%), `Fence` (80.5%) — tỷ lệ missing quá cao không phải do lỗi ghi chép mà vì đại đa số nhà thực sự **không có** những tiện ích này. Đây là **MNAR (Missing Not At Random)** có cấu trúc.

Tuy nhiên thông tin "có hay không" vẫn có giá trị dự báo, nên trước khi drop cần tạo biến cờ: `Has_Pool`, `Has_Alley`, `Has_Fence`, `Has_Misc_Feature`.

**Kỹ thuật: Tạo cờ binary → Drop cột gốc.**

---

#### Nhóm 3 — Garage: MNAR có cấu trúc + MAR

Khi xác định nhà có garage hay không, **không thể chỉ dựa vào một cột**. Đối chiếu chéo 3 cột (`Garage Type`, `Garage Area`, `Garage Cars`) cho thấy 157 nhà không có garage, và tất cả biến garage của nhóm này đều null đồng bộ — đây là **MNAR**: nhà không có garage thì không có năm xây, diện tích, chất lượng garage.

Vấn đề điển hình: nếu áp dụng khuôn mẫu "biến số → điền median", `Garage Yr Blt` của 157 nhà không có garage sẽ được điền ~1979 — mâu thuẫn hoàn toàn với `Garage Type = 'None'`. Mô hình OLS sẽ nhận tín hiệu nhiễu: nhà không có garage nhưng lại có năm xây garage.

**Các trường hợp cần xử lý:**

| Trường hợp | Số lượng | Cơ chế | Kỹ thuật |
|------------|----------|--------|-----------|
| Không có garage | 157 hàng | MNAR | Gán `'None'` cho `Garage Type` và các biến categorical, `0` cho `Garage Cars` và `Garage Area`, giữ `NaN` cho `Garage Yr Blt` |
| Có garage, thiếu `Garage Yr Blt` | 2 hàng | MAR | Median theo nhóm (thập niên `Year Built` × `Garage Type`), fallback theo `Garage Type`, fallback toàn cục |
| Có garage, thiếu `Garage Area` | 1 hàng | MAR | Median theo `Garage Type` |
| Có garage, thiếu `Garage Cars` | 1 hàng | MAR | Median theo nhóm diện tích `Garage Area` (chỉ dùng dòng có `Garage Cars > 0` và `Garage Area > 0` để tính bins) |
| Có garage, thiếu `Garage Type` (do `Area/Cars > 0` nhưng `Type` bỏ trống) | edge case | MAR | Mode theo nhóm diện tích `Garage Area` (chỉ dùng dòng có `Garage Type` đã biết để tính bins, tránh index mismatch) |
| `Garage Type = 'Attchd'` có `Garage Yr Blt < Year Built` | 5 hàng | Lỗi nhập liệu | Sửa `Garage Yr Blt = Year Built` (garage đính kèm không thể xây trước nhà) |
| `Garage Finish`, `Garage Qual`, `Garage Cond` | 159 hàng mỗi biến | MAR | Mode theo nhóm (`Garage Type` × `Overall Qual`), fallback theo `Garage Type` |

**Kỹ thuật xử lý `Garage Yr Blt`:**
- Nhà không có garage: giữ `NaN` (không gán giá trị vì feature không tồn tại).
- Nhà có garage nhưng missing (2 hàng MAR): median theo nhóm thập niên × loại garage.
- Sau xử lý, `Garage Yr Blt` chỉ còn `NaN` ở nhóm không có garage — sẽ được xử lý tiếp trong Feature Engineering (tạo `Has_Garage` và `Garage_Age`).
- Kiểm tra thêm ở Bước 5: nếu `Garage Yr Blt > Yr Sold` (lỗi nhập liệu rõ ràng) thì gán lại bằng `Year Built`.

---

#### Nhóm 4 — Basement: MNAR + MAR

80 nhà không có tầng hầm (anchor: `Total Bsmt SF = 0` hoặc tổng diện tích các phần = 0). Toàn bộ biến basement của nhóm này đều là **MNAR**.

Tuy nhiên có một số hàng null trong khi nhà **có** tầng hầm — đây là **MAR** (lỗi bỏ sót khi ghi chép), cần xử lý riêng.

**Lưu ý thứ tự xử lý:** `Total Bsmt SF` và `Bsmt Unf SF` có quan hệ kế toán (`Total = Fin1 + Fin2 + Unf`). Phải xử lý theo đúng thứ tự nhân quả:
1. Tính `Total Bsmt SF` trước (từ tổng các phần, dùng `fillna(0)` để tránh NaN propagation)
2. Tính `Bsmt Unf SF` sau (phần dư = `Total - Fin1 - Fin2`)

Đảo thứ tự sẽ gây NaN propagation: `NaN - 500 - 0 = NaN`.

| Biến                                         | Số lượng MAR | Kỹ thuật |
|----------------------------------------------|--------------|--------|
| `Bsmt Exposure`                              | 3 hàng       | Mode theo `BsmtFin Type 1` của nhà có tầng hầm (`notna()` filter); fallback global mode nếu Type không có trong lookup |
| `BsmtFin Type 2`                             | 1 hàng       | Mode của nhà có `BsmtFin SF 2 > 0` |
| `BsmtFin SF 1`                               | 1 hàng       | Median theo `BsmtFin Type 1` (chỉ dùng dòng có `SF > 0`) |
| `BsmtFin SF 2`                               | 1 hàng       | Median theo `BsmtFin Type 2` (chỉ dùng dòng có `SF > 0`) |
| `BsmtFin Type 1/2` (SF = 0, Type null)       | một số hàng | Gán thẳng `'Unf'` — diện tích = 0 thì loại hoàn thiện chắc chắn là chưa hoàn thiện (logic, không phải imputation) |
| `BsmtFin Type 1/2` (biết SF > 0, thiếu Type) | một số hàng  | Mode theo nhóm diện tích SF (bins tính từ dòng có `SF > 0` và có tầng hầm, tránh giá trị 0 kéo lệch ranh giới nhóm) |
| `Bsmt Unf SF`                                | 1 hàng       | Tính từ phần dư: `Total Bsmt SF - Fin1 - Fin2` (quan hệ kế toán) |
| `Total Bsmt SF`                              | 1 hàng       | Tính từ tổng: `Fin1 + Fin2 + Unf` |
| `Bsmt Full Bath`                             | 2 hàng       | Điền `0` (tầng hầm không bắt buộc có phòng tắm) |
| `Bsmt Half Bath`                             | 2 hàng       | Điền `0` |
| `Bsmt Qual`                                  | 80 hàng      | Mode theo nhóm quantile `Total Bsmt SF` (tương quan r=0.46); bins tính từ dòng có tầng hầm và `SF > 0` |
| `Bsmt Cond`                                  | 80 hàng      | Mode theo thập niên `Year Built` — nhà trước 1900 có tỷ lệ `'Fa'` lên đến 28–33%, khác biệt đáng kể so với global mode 89% là `'TA'`; fallback `'TA'` |

---

#### Nhóm 5 — Fireplace Qu: MNAR thuần túy

1,422 hàng (48.5%) `Fireplace Qu = NaN` khi `Fireplaces = 0` — 100% khớp, không có ngoại lệ. Không có lò sưởi thì không có chất lượng lò sưởi. Đây là **MNAR** rõ ràng nhất trong dataset.

Kiểm tra data thực tế xác nhận: không có hàng nào `Fireplaces > 0` mà `Fireplace Qu = NaN`, do đó **không cần imputation cho nhóm MAR**.

**Kỹ thuật: `Fireplaces = 0` → điền `'None'`.** Pipeline được thiết kế phòng thủ (defensive): nếu trong tương lai xuất hiện trường hợp MAR (nhà có lò sưởi nhưng thiếu `Fireplace Qu`), tự động điền mode theo nhóm `(Overall Qual, Bldg Type)` — phản ánh thực tế là chất lượng lò sưởi phụ thuộc vào chất lượng tổng thể và loại nhà.

---

#### Nhóm 6 — Mas Vnr (Lớp ốp đá/gạch): MNAR + MAR nhỏ

1,775 hàng (60.6%) `Mas Vnr Type = NaN` — phần lớn là nhà không có lớp ốp đá (**MNAR**). Anchor được xác định bằng đối chiếu chéo: `Mas Vnr Area > 0` HOẶC `Mas Vnr Type ≠ NaN và ≠ 'None'`.

7 hàng có diện tích ốp đá > 0 nhưng thiếu loại (`Mas Vnr Type = NaN`) — **MAR** (lỗi nhập liệu). Không thể điền mode toàn tập vì phân phối loại ốp đá phụ thuộc diện tích: diện tích nhỏ thường là `BrkFace`, lớn thường là `Stone`. Bins cố định `[0, 100, 300, 10000]` được chọn dựa trên đặc tính vật lý của từng loại vật liệu.

**Kỹ thuật:**
- MNAR (1,768 hàng): điền `'None'` / `0`
- MAR — 7 hàng thiếu `Mas Vnr Type`: mode theo nhóm diện tích (small/medium/large)
- Thiếu `Mas Vnr Area` khi có ốp đá: không xuất hiện trong dataset này (0 hàng); pipeline xử lý phòng thủ bằng median theo loại `Mas Vnr Type` nếu gặp ở dữ liệu khác

---

#### Nhóm 7 — Lot Frontage: MAR

490 hàng (16.7%) missing. Mọi lô đất đều có mặt tiền — việc thiếu là do **bỏ sót khi đo đạc** → **MAR**. Bằng chứng: tỷ lệ missing thay đổi rõ rệt theo khu vực (`GrnHill`: 100%, `Landmrk`: 100%, `ClearCr`: 54.5%, trong khi nhiều khu vực khác <10%).

Quan trọng hơn, median `Lot Frontage` dao động rất lớn theo neighborhood: từ 21 ft (`BrDale`) đến 92 ft (`NridgHt`), trong khi global median chỉ là 68 ft. Dùng global median sẽ sai lệch lớn cho các khu vực đặc thù.

**Kỹ thuật:** Grouped median theo `Neighborhood`. Fallback bằng global median cho những dòng vẫn missing sau grouped median — thực tế có **3 dòng** phải dùng global median fallback (thuộc các neighborhood quá ít mẫu để tính median).

---

#### Nhóm 8 — Electrical: MCAR

Chỉ 1 hàng missing (0.03%), không có quy luật kết hợp với bất kỳ biến nào — **MCAR** thuần túy. `SBrkr` chiếm 91.5% giá trị.

**Kỹ thuật: Điền mode = `'SBrkr'`.**

---

#### Tóm tắt cơ chế và kỹ thuật

| Nhóm | Biến | Missing | Cơ chế | Kỹ thuật |
|------|------|---------|---------|-----------|
| Định danh / NZV | `Order`, `PID`, `Utilities`, `Street` | — | — | Drop |
| >80% missing | `Pool QC`, `Misc Feature`, `Alley`, `Fence` | 2358–2917 | MNAR | Tạo cờ → Drop |
| Garage (MNAR) | Garage cluster | 157–159 | MNAR | `'None'` / `0` (giữ `NaN` cho `Garage Yr Blt`) |
| Garage (MAR) | `Garage Yr Blt` | 2 | MAR | Median theo thập niên × loại, fallback cascade |
| Garage (MAR) | `Garage Area` | 1 | MAR | Median theo `Garage Type` |
| Garage (MAR) | `Garage Cars` | 1 | MAR | Median theo nhóm diện tích `Garage Area` |
| Garage (MNAR) | `Garage Type` | 157 | MNAR | `'None'` (toàn bộ 157 hàng là nhà không có garage) |
| Garage (MAR) | `Garage Type` (có `Area/Cars > 0` nhưng `Type` bỏ trống) | edge case | MAR | Mode theo nhóm diện tích `Garage Area` |
| Garage quality | `Finish`, `Qual`, `Cond` | 159 | MAR | Mode theo `Garage Type` × `Overall Qual`, fallback cascade |
| Basement (MNAR) | Basement cluster | 80–83 | MNAR | `'None'` / `0` |
| Basement (MAR) | `Bsmt Exposure` | 3 | MAR | Mode theo `BsmtFin Type 1` của nhà có tầng hầm; fallback global mode |
| Basement (MAR) | `BsmtFin Type 2` | 1 | MAR | Mode nhà có `BsmtFin SF 2 > 0` |
| Basement (MAR) | `BsmtFin SF 1/2` | 1 | MAR | Median theo loại Type (chỉ dùng `SF > 0`) |
| Basement | `BsmtFin Type 1/2` (SF = 0, Type null) | vài hàng | Logic | Gán thẳng `'Unf'` (không phải imputation) |
| Basement (MAR) | `BsmtFin Type 1/2` (SF > 0, thiếu Type) | vài hàng | MAR | Mode theo nhóm diện tích SF |
| Basement | `Total Bsmt SF` | 1 | Kế toán | Tính từ `Fin1 + Fin2 + Unf` (xử lý **trước** `Bsmt Unf SF`) |
| Basement | `Bsmt Unf SF` | 1 | Kế toán | Tính từ `Total - Fin1 - Fin2` (xử lý **sau** `Total Bsmt SF`) |
| Basement | `Bsmt Full/Half Bath` | 2 | MAR | Điền `0` |
| Basement | `Bsmt Qual` | 80 | MAR | Mode theo nhóm quantile `Total Bsmt SF` |
| Basement | `Bsmt Cond` | 80 | MAR | Mode theo thập niên `Year Built`, fallback `'TA'` |
| Fireplace (MNAR) | `Fireplace Qu` | 1422 | MNAR | `'None'` (không cần imputation thêm) |
| Mas Vnr (MNAR) | `Mas Vnr Type` | 1775 | MNAR | `'None'` / `0` |
| Mas Vnr (MAR) | `Mas Vnr Type` | 7 | MAR | Mode theo nhóm diện tích |
| Mas Vnr (MAR) | `Mas Vnr Area` (thiếu Area khi có ốp đá) | 0 (dataset này) | MAR | Median theo loại Type (defensive) |
| Lot Frontage (MAR) | `Lot Frontage` | 490 | MAR | Grouped median theo `Neighborhood`; fallback global median (3 dòng) |
| Electrical (MCAR) | `Electrical` | 1 | MCAR | Mode (`SBrkr`) |

Dựa trên phân tích cơ chế ở trên, nhóm triển khai xử lý missing values trong module `data_pipeline.py`. Toàn bộ logic được xây dựng theo **quan hệ nhân-quả** giữa các biến.

Quy trình tiền xử lí có 6 bước tuần tự: xóa biến vô dụng → kiểm tra tính nhất quán → xử lý missing theo từng nhóm → imputation Lot Frontage & Electrical → tạo biến mới → kiểm tra lần cuối (raise lỗi nếu còn missing).

# Phần 3: Data Pipeline và VIF



## 3.1. Feature Engineering (Kỹ nghệ Đặc trưng)

Quá trình tạo biến mới được thực hiện ở **hai tầng** tách biệt nhau về mặt kiến trúc:

**Tầng 1 — Tiền xử lý thô (trước Pipeline):** Các biến được tạo trực tiếp trên DataFrame gốc tại Bước 5 của `clean_data()`, vì chúng phụ thuộc vào nhiều cột gốc cần xóa ngay sau đó để tránh đa cộng tuyến.

**Tầng 2 — Bên trong DataPipeline:** Các biến tương tác và tổng hợp bậc cao được tạo trong `_create_new_features()` và `_create_interactions()` sau khi dữ liệu đã được chuẩn hóa, đảm bảo không bị data leakage.

| Tầng | Biến mới | Ý nghĩa |
|---|---|---|
| Thô | `Age_At_Sale`, `Remod_Age`, `Was_Remodeled` | Tuổi nhà và lịch sử cải tạo |
| Thô | `Garage_Age`, `Has_Garage` | Tuổi và sự tồn tại của garage |
| Thô | `Total_Bath`, `Total_Porch` | Tổng hợp không gian |
| Thô | `Has_Low_Qual_Fin`, `Is_Normal_Sale`, `Has_Negative_Condition` | Cờ nhị phân hóa |
| Pipeline | `Total_SqFt` | Tổng diện tích sử dụng (tầng nổi + tầng hầm) |
| Pipeline | `Qual_x_GrLivArea` | Tương tác chất lượng × diện tích sống |

## 3.2. Lựa chọn đặc trưng (Feature Selection & Dimension Reduction)

Sau khi tạo biến tổng hợp, các biến gốc được loại bỏ ngay lập tức để tránh **đa cộng tuyến hoàn hảo** và giảm số chiều dữ liệu:

| Nhóm | Biến bị xóa | Lý do |
|---|---|---|
| Thời gian | `Year Built`, `Year Remod/Add`, `Yr Sold` | Đã hấp thụ hoàn toàn vào `Age_At_Sale`, `Remod_Age`, `Was_Remodeled` |
| Garage | `Garage Yr Blt` | Thay bằng `Garage_Age` và `Has_Garage`; loại `NaN` MNAR khỏi mô hình |
| Phòng tắm | `Full Bath`, `Half Bath`, `Bsmt Full Bath`, `Bsmt Half Bath` | Gộp vào `Total_Bath` với trọng số (Half Bath = 0.5) |
| Hiên nhà | `Open Porch SF`, `Enclosed Porch`, `3Ssn Porch`, `Screen Porch` | Gộp vào `Total_Porch` |
| Cờ nhị phân | `Low Qual Fin SF`, `Condition 1`, `Condition 2` | Chuyển hóa thành biến cờ tổng hợp mang tính đại diện cao hơn |

## 3.3. Kiến Trúc DataPipeline

Hệ thống xử lý dữ liệu được chia làm hai giai đoạn: **Tiền xử lý thô (Xóa dòng ngoại lai)** và **Pipeline biến đổi đặc trưng (Fit → Transform)**. Thiết kế này đảm bảo các tham số biến đổi chỉ được học từ tập train và không gây ra lỗi bất đồng bộ kích thước giữa ma trận đặc trưng $X$ và vector mục tiêu $y$.

### Giai đoạn 1: Tiền xử lý dữ liệu thô (Trước Pipeline)
* **Outlier Removal:** Sử dụng phương pháp IQR để phát hiện và loại bỏ đồng thời các dòng chứa giá trị ngoại lai trên cả tập $X\_train$ và $y\_train$ trước khi đưa vào mô hình (ví dụ: lọc bỏ các mẫu có diện tích hoặc giá trị bất thường).

### Giai đoạn 2: Quy trình DataPipeline (Kiến trúc Fit → Transform)
Sau khi dữ liệu đã được làm sạch số dòng, quy trình biến đổi gồm **6 bước tuần tự** trong Pipeline:

| Bước | Tên bước | Mô tả |
|------|----------|-------|
| A | **Outlier Smoothing** | Winsorize (cắt 5% mỗi đuôi) để làm mượt các biến số còn lại |
| B | **Feature Engineering** | Tạo `Total_SqFt` = `Gr Liv Area` + `Total Bsmt SF` |
| C | **Ordinal Encoding** | Map thứ bậc chất lượng (`Ex→5`, `Gd→4`, `TA→3`, `Fa→2`, `Po→1`, `None→0`) |
| D | **One-Hot Encoding** | Mã hóa 20 biến danh nghĩa (Neighborhood, BldgType...), `drop_first=True` |
| E | **Log Transform** | `log1p` các cột số có skew > 0.75 (học trên train, apply cả test) |
| F | **Z-score Standardization** | Chuẩn hóa toàn bộ cột số về μ=0, σ=1 |

**Lưu ý chống Data Leakage:** `train_test_split` được thực hiện **trước** khi thực hiện bất kỳ bước tiền xử lý hay gọi `fit_transform nào`, đảm bảo phân phối của test set hoàn toàn độc lập.

In [ ]:

# ── Load & clean ────────────────────────────────────────────────────
df_clean = clean_data(DATA_PATH, verbose=False)
print(f'Shape sau clean_data: {df_clean.shape}')

# ── Tách target ─────────────────────────────────────────────────────
y_raw = df_clean['SalePrice']
X_raw = df_clean.drop(columns=['SalePrice'])

# ── Train / Test split 80/20 — TRƯỚC pipeline để tránh leakage ──────
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split_df(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# ── SỬA LỖI Ở ĐÂY: Biến đổi Log cho biến mục tiêu (y) ───────────────
y_train_log = np.log1p(y_train_raw)
y_test_log = np.log1p(y_test_raw)
# ────────────────────────────────────────────────────────────────────

print(f'Train : {X_train_raw.shape[0]} mẫu  |  Test : {X_test_raw.shape[0]} mẫu')
print(f'Số features gốc: {X_raw.shape[1]}')

## 3.4. Chạy DataPipeline

Pipeline được cấu hình với các tham số sau:

- **`outlier_method='winsorize'`, `outlier_threshold=0.05`:** Cắt 5% mỗi đuôi cho cả biến liên tục lẫn target. Phương pháp Winsorization được ưu tiên hơn Remove vì bảo toàn số hàng — quan trọng với tập dữ liệu ~2,930 quan sát.
- **`encoding='auto'`:** Tự động phân loại Ordinal/One-hot dựa trên `ORDINAL_MAPS` và `NOMINAL_COLS` đã định nghĩa trong `data_pipeline.py`.
- **`log_target=False`:** Tắt tính năng này trong Pipeline do SalePrice đã được log1p thủ công trước (y_train_log = np.log1p(y_train_raw)), tránh double-log.
- **`log_skewed_features=True`:** Pipeline tự động phát hiện và log1p các cột số có skew > 0.75 trên tập train.
- **`engineer_features=True`:** Bật tạo biến tổng hợp (Total_SqFt, Qual_x_GrLivArea) bên trong Pipeline. 
- **`add_interactions=False`:** Tắt Interaction — các mô hình Polynomial/Interaction sẽ tự tạo ở Phần 4.


In [ ]:
pipe = DataPipeline(
    outlier_method      = 'winsorize',
    outlier_threshold   = 0.05,
    encoding            = 'auto',
    scale               = True,
    log_target          = False, 
    engineer_features   = True,  # Đã sửa thành True theo yêu cầu review
    log_skewed_features = True,
    add_interactions    = False, # Giữ nguyên False
)

X_train, y_train = pipe.fit_transform(X_train_raw, y_train_log) 
X_test           = pipe.transform(X_test_raw)

print(f'\nShape X_train sau pipeline : {X_train.shape}')
print(f'Shape X_test  sau pipeline : {X_test.shape}')
print(f'Số cột log-transformed     : {len(pipe._skewed_cols_to_log)}')
print(f'Số cột ordinal encoded     : {len(pipe._ordinal_cols_)}')
print(f'Số cột one-hot encoded     : {len(pipe._onehot_cols_)}')
print(f'Số dummy columns tạo ra   : {len(pipe._dummy_cols_)}')
pipe.summary()

## 3.5. Phân Tích VIF và Khử Đa Cộng Tuyến

**VIF (Variance Inflation Factor)** đo mức độ đa cộng tuyến của từng biến so với các biến còn lại. Quy tắc ngưỡng:

| Giá trị VIF | Mức độ | Quyết định |
|-------------|--------|------------|
| VIF < 5 | Không có đa cộng tuyến | Giữ lại |
| 5 ≤ VIF < 10 | Đa cộng tuyến trung bình | Theo dõi |
| VIF > 10 | Đa cộng tuyến nghiêm trọng | **Loại bỏ** |
| VIF = ∞ | Đa cộng tuyến hoàn hảo (Perfect MC) | **Loại bỏ ngay** |

Pipeline sử dụng thuật toán **lặp (iterative elimination):** mỗi vòng lặp tính lại VIF cho tất cả cột còn lại, loại cột có VIF cao nhất, rồi lặp lại cho đến khi mọi cột thỏa mãn `VIF ≤ 10`. Phương pháp này chính xác hơn loại một lần vì VIF của một cột thay đổi khi cột khác bị loại.

**Kết nối với phân tích EDA (phần 2.3):** Clustermap đã cảnh báo trước các nhóm cột đa cộng tuyến cao (`Garage Cars` & `Garage Area`, `TotRms AbvGrd` & `Gr Liv Area`, v.v.). Bước VIF ở đây là sự xác nhận định lượng cho các quan sát trực quan đó.


In [ ]:
print('VIF TRƯỚC KHI LỌC (Toàn bộ các cột có VIF >= 10):')
vif_before = run_vif_check(X_train)

# 1. Chỉ lọc lấy những dòng có VIF vượt ngưỡng 10
vif_loi = vif_before[vif_before['VIF'] >= 10]

# 2. Sắp xếp giảm dần để thấy rõ những "thủ phạm" nặng nhất ở trên cùng
vif_loi_sorted = vif_loi.sort_values(by='VIF', ascending=False)

# 3. Dùng option_context để ép Jupyter hiển thị tất cả các dòng mà không bị ẩn "..."
with pd.option_context('display.max_rows', None):
    display(vif_loi_sorted)

# Thống kê lại
n_high_vif = len(vif_loi)
n_inf_vif  = np.isinf(vif_loi['VIF']).sum()
print(f'\nTổng cột VIF >= 10  : {n_high_vif}')
print(f'Trong đó VIF = inf  : {n_inf_vif}')

In [ ]:
# ── Chạy iterative VIF elimination ───────────────────────────────
print('\nBắt đầu khử đa cộng tuyến (threshold=10)...')
print('─' * 60)

X_train, dropped_cols = pipe.drop_high_vif(X_train, y_train, threshold=10)  # ← thêm y_train

# [ĐÃ FIX]: Đồng bộ test set bằng cách xóa trực tiếp các cột, không chạy lại toàn bộ transform
# Dùng errors='ignore' để phòng trường hợp chạy lại cell nhiều lần không bị lỗi
X_test = X_test.drop(columns=dropped_cols, errors='ignore')

print(f'\nX_train sau VIF : {X_train.shape}')
print(f'X_test  sau VIF : {X_test.shape}')
print(f'\nDanh sách {len(dropped_cols)} cột bị loại:')
for i, col in enumerate(dropped_cols, 1):
    print(f'  {i:>2}. {col}')

In [ ]:
# ── Tính VIF sau khi lọc — kiểm tra không còn cột nào vượt ngưỡng ──
print('VIF SAU KHI LỌC (Các cột vẫn còn VIF >= 10 nếu bị sót):')
vif_after = run_vif_check(X_train)

# 1. Lọc và sắp xếp các cột vẫn còn VIF >= 10
vif_still_high_df = vif_after[vif_after['VIF'] >= 10].sort_values(by='VIF', ascending=False)

# 2. Hiển thị toàn bộ danh sách lỗi không bị cắt dòng
with pd.option_context('display.max_rows', None):
    display(vif_still_high_df)

# 3. Đếm số lượng và kiểm tra chốt chặn
still_high = len(vif_still_high_df)
print(f'\nSố cột còn VIF >= 10: {still_high}  ← Phải bằng 0')

# Dừng chương trình nếu vẫn còn cột VIF cao
assert still_high == 0, 'Vẫn còn cột VIF cao! Kiểm tra lại pipeline.'
print('✅ Tất cả cột đều VIF < 10. Pipeline sẵn sàng cho huấn luyện.')


#### Nhận xét:

- Chiến lược loại bỏ biến hợp lý: Quy trình xử lý không chỉ dựa trên ngưỡng VIF mà còn kết hợp đánh giá mức độ tương quan của từng biến với biến mục tiêu. Các đặc trưng có khả năng dự báo thấp được ưu tiên loại bỏ trước, giúp giảm đa cộng tuyến trong khi vẫn duy trì những thông tin quan trọng cho mô hình.

- Xử lý hiệu quả đa cộng tuyến nghiêm trọng: Thuật toán đã phát hiện và loại bỏ thành công các biến có giá trị `VIF = ∞`, phản ánh hiện tượng phụ thuộc tuyến tính hoàn hảo giữa các đặc trưng. Điều này giúp tránh các vấn đề về ma trận suy biến và nâng cao độ ổn định của mô hình.

- Loại bỏ hiện tượng Dummy Variable Trap: Phần lớn các biến bị loại là các biến giả (dummy variables) được tạo ra từ quá trình One-Hot Encoding. Kết quả này cho thấy quy trình lọc đã xử lý tốt sự chồng chéo thông tin giữa các nhóm biến phân loại, từ đó giảm đáng kể mức độ đa cộng tuyến trong dữ liệu.

- Loại bỏ các đặc trưng dư thừa: Một số biến có tương quan khá cao với biến mục tiêu vẫn được loại bỏ do chứa thông tin trùng lặp với các đặc trưng khác. Điều này giúp giữ lại tập đặc trưng gọn hơn nhưng vẫn đảm bảo khả năng biểu diễn thông tin của dữ liệu.

- Nâng cao chất lượng dữ liệu đầu vào: Việc loại bỏ 32 đặc trưng dư thừa giúp giảm số chiều dữ liệu, tăng tính ổn định của các hệ số hồi quy, hạn chế nguy cơ overfitting và cải thiện hiệu quả huấn luyện của các mô hình học máy.

#### Kết luận:

Bước xử lý đa cộng tuyến đã làm sạch dữ liệu một cách hiệu quả, loại bỏ các đặc trưng dư thừa và các mối quan hệ phụ thuộc tuyến tính không cần thiết. Sau khi xử lý, toàn bộ các biến còn lại đều đạt ngưỡng `VIF ≤ 10`, đồng thời hai tập `X_train` và `X_test` được đồng bộ hoàn toàn về số lượng đặc trưng, tạo nền tảng ổn định cho giai đoạn xây dựng và đánh giá mô hình.

## 3.6. Trực Quan Hóa VIF Trước và Sau Lọc


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Phân phối VIF Trước và Sau Khử Đa Cộng Tuyến', fontsize=15, fontweight='bold')

for ax, (vif_df, title, color) in zip(axes, [
    (vif_before.head(30), 'VIF Trước Lọc (Top 30)', '#E57373'),
    (vif_after.head(30),  'VIF Sau Lọc (Top 30)',   '#81C784'),
]):
    vals = vif_df['VIF'].replace([np.inf, -np.inf], np.nan).fillna(999)
    bars = ax.barh(vif_df['feature'], vals, color=color, edgecolor='white', linewidth=0.5)

    # Vẽ đường ngưỡng VIF=10
    ax.axvline(x=10, color='red', linestyle='--', linewidth=1.5, label='Ngưỡng VIF = 10')

    ax.set_xlabel('VIF Score', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.legend(fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    
    # --- ĐOẠN CODE ĐÃ FIX ---
    # 1. Tính mức trần động (không vượt quá 50 để tránh outlier kéo giãn biểu đồ 1)
    upper_bound = min(vals.max() * 1.1, 50)
    
    # 2. Đảm bảo giới hạn phải lớn hơn hoặc bằng 12 (để vạch đỏ x=10 luôn lọt vào giữa biểu đồ 2)
    ax.set_xlim(0, max(upper_bound, 12))
    # -------------------------

plt.tight_layout()
plt.show()

print(f'Số features trước VIF lọc : {len(vif_before)}')
print(f'Số features sau  VIF lọc  : {len(vif_after)}')
print(f'Số features bị loại       : {len(vif_before) - len(vif_after)}')

#### Nhận xét

- **Giảm mạnh mức độ đa cộng tuyến:** Trước khi xử lý, nhiều biến có giá trị VIF rất cao, vượt xa ngưỡng an toàn **VIF = 10**, thậm chí xuất hiện các trường hợp **VIF = INF**, cho thấy tồn tại đa cộng tuyến nghiêm trọng trong tập dữ liệu. Sau khi áp dụng quy trình lọc, toàn bộ các biến còn lại đều có **VIF < 10**, chứng minh đa cộng tuyến đã được kiểm soát hiệu quả.

- **Hiệu quả loại bỏ đặc trưng dư thừa:** Số lượng đặc trưng giảm từ **208** xuống còn **176**, tương ứng loại bỏ **32 biến** có tính phụ thuộc cao. Điều này cho thấy pipeline đã loại bỏ thành công các biến dư thừa mà vẫn duy trì phần lớn thông tin của bộ dữ liệu.

- **Cải thiện chất lượng tập đặc trưng:** Phân phối VIF sau xử lý tập trung ở mức thấp hơn đáng kể so với trước, cho thấy mức độ phụ thuộc tuyến tính giữa các biến đã giảm rõ rệt. Điều này giúp các hệ số hồi quy ổn định hơn và nâng cao khả năng diễn giải của mô hình.

- **Giữ lại các biến có giá trị dự báo:** Mặc dù đã loại bỏ nhiều đặc trưng, một số biến quan trọng vẫn được giữ lại với VIF ở mức trung bình (**5–9**). Đây là mức chấp nhận được trong thực tế, giúp cân bằng giữa việc giảm đa cộng tuyến và bảo toàn thông tin phục vụ dự đoán.

#### Kết luận

Biểu đồ phân phối VIF trước và sau xử lý cho thấy bước khử đa cộng tuyến đã đạt hiệu quả cao. Tập dữ liệu sau xử lý có cấu trúc ổn định hơn, giảm đáng kể sự phụ thuộc giữa các biến và sẵn sàng cho quá trình huấn luyện các mô hình học máy.

## 3.7. Kiểm Tra Phân Phối Dữ Liệu Sau Pipeline

Sau khi qua Pipeline, dữ liệu đã được:
1. **Winsorize** — cắt bớt ảnh hưởng của các điểm đòn bẩy cực đoan
2. **Log1p transform** — kéo phân phối lệch phải về gần chuẩn hơn
3. **Z-score standardization** — đưa mọi cột về cùng đơn vị đo (μ=0, σ=1)

Bước kiểm tra dưới đây xác nhận target `y_train` (đã log1p) có phân phối gần chuẩn hơn so với `SalePrice` gốc — điều kiện cần cho giả định phần dư OLS.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phân phối Target: Trước và Sau log1p Transform', fontsize=14, fontweight='bold')

# ── Trái: SalePrice gốc ─────────────────────────────────────────
ax = axes[0]

sns.histplot(y_train_raw, kde=True, stat='density', color='salmon', bins=50, ax=ax)
mu0, std0 = norm.fit(y_train_raw)
x0 = np.linspace(y_train_raw.min(), y_train_raw.max(), 200)
ax.plot(x0, norm.pdf(x0, mu0, std0), 'k--', lw=2, label=f'Normal (μ={mu0:.0f})')
ax.set_title('SalePrice Gốc (Trước pipeline)', fontsize=12)
ax.set_xlabel('SalePrice (USD)')
ax.legend()
ax.spines[['top','right']].set_visible(False)

# ── Phải: log1p(SalePrice) ──────────────────────────────────────
ax = axes[1]
sns.histplot(y_train, kde=True, stat='density', color='steelblue', bins=50, ax=ax)
mu1, std1 = norm.fit(y_train)
x1 = np.linspace(float(y_train.min()), float(y_train.max()), 200)
ax.plot(x1, norm.pdf(x1, mu1, std1), 'k--', lw=2, label=f'Normal (μ={mu1:.3f})')
ax.set_title('log1p(SalePrice) Sau Pipeline', fontsize=12)
ax.set_xlabel('log1p(SalePrice)')
ax.legend()
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

print('Thống kê so sánh:')
print(f'  SalePrice gốc     — Skewness: {skew(y_train_raw):.3f} | Kurtosis: {kurtosis(y_train_raw):.3f}')
print(f'  log1p(SalePrice)  — Skewness: {skew(y_train):.3f} | Kurtosis: {kurtosis(y_train):.3f}')


#### Nhận xét: Kết quả sau Data Pipeline & Bộ lọc Đa cộng tuyến Nâng cao

##### 1. Biến đổi Biến Mục tiêu (Target Transformation)
Biểu đồ phân phối trước và sau cho thấy phép biến đổi `log1p` đã tối ưu hóa biến mục tiêu `SalePrice` rất hiệu quả:
- Chỉ số **Skewness** giảm mạnh từ **1.749** xuống **0.188**, tiệm cận mức **0** lý tưởng.
- Chỉ số **Kurtosis** giảm từ **5.460** xuống **-0.640**.
- Phân phối dữ liệu sau biến đổi trở nên cân đối hơn, giảm hiện tượng lệch phải và hạn chế ảnh hưởng của các giá trị ngoại lệ.

Nhờ đó, dữ liệu đầu ra đáp ứng tốt hơn các giả định của các mô hình hồi quy tuyến tính, đặc biệt là giả định về tính chuẩn của phần dư (Residual Normality).

##### 2. Cơ chế Lọc VIF Thông minh kết hợp Ý nghĩa Thống kê (3-Tier Sorting Algorithm)
Thay vì loại bỏ biến dựa trên thứ tự xuất hiện trong tập dữ liệu, hàm `drop_high_vif()` đã được nâng cấp thành một cơ chế xếp hạng đa tầng có xét đến thông tin từ biến mục tiêu `y`. 

Trong mỗi vòng lặp:
1. Hệ thống xác định các biến có mức đa cộng tuyến cao (`VIF > 10`).
2. Một mô hình OLS phụ trợ được xây dựng để:
   - Tính **P-value** của từng biến.
   - Đánh giá mức độ tương quan tuyệt đối với biến mục tiêu.
3. Thuật toán ưu tiên loại bỏ:
   - Biến có **P-value cao nhất** (ít ý nghĩa thống kê nhất), hoặc
   - Biến có **độ tương quan thấp nhất** với giá nhà.

Nhờ cơ chế này:
- Các đặc trưng quan trọng chứa nhiều thông tin dự báo được giữ lại.
- Các biến giả (dummy variables) dư thừa được loại bỏ hợp lý.
- Các cặp biến mang thông tin gần như trùng lặp (như `Garage Area` và `Garage Cars`, `1st Flr SF` và `Total Bsmt SF`) được xử lý một cách có cơ sở thống kê thay vì loại bỏ ngẫu nhiên.

##### 3. Bảo toàn Cấu trúc Dữ liệu và Phòng chống Data Leakage
Sau khi hoàn tất quá trình lọc VIF lặp nhiều vòng:
- Ma trận đặc trưng `X_train` được tinh gọn đáng kể.
- Toàn bộ các biến còn lại đều có **VIF < 10**, đảm bảo mức đa cộng tuyến nằm trong ngưỡng chấp nhận được.
- Danh sách các biến bị loại (`dropped_cols`) được lưu lại từ tập huấn luyện.

Danh sách này sau đó được áp dụng đồng bộ lên tập kiểm tra, đảm bảo không xảy ra hiện tượng rò rỉ dữ liệu (Data Leakage).

# Phần 4: Đánh Giá Mô Hình Trên Test Set — Ames Housing



**Các mô hình được đánh giá (18 mô hình):**
1. OLS Basic
2. Polynomial Features + OLS Basic 
3. Interaction Features + OLS Basic
4. OLS Feature Selection (VIF + p-value)
5. Polynomial Features + OLS Feature Selection (VIF + p-value)
6. Interaction Features + OLS Feature Selection (VIF + p-value)
7. Ridge CV
8. Polynomial Features + Ridge CV
9. Interaction Features + Ridge CV
10. Lasso CV
11. Polynomial Features + Lasso CV
12. Interaction Features + Lasso CV
13. Kernel Ridge Regression (RBF)
14. Polynomial Features + Kernel Ridge Regression (RBF)
15. Interaction Features + Kernel Ridge Regression (RBF)
16. Bayesian Linear Regression
17. Polynomial Features + Bayesian Linear Regression
18. Interaction Features + Bayesian Linear Regression

## 4.1. Huấn luyện & Đánh giá tất cả mô hình

In [ ]:
y_test_usd = y_test_raw.copy()          # USD-space chính là giá gốc chưa biến đổi
y_test_log = np.log1p(y_test_raw)       # Log-space (dùng log1p để đồng nhất với np.expm1 trong hàm của bạn)
# ── Hàm tiện ích: đánh giá trên test set ──────────────────────────────
def eval_on_test(model_name, y_pred_log):
    # 1. Log-space (Không can thiệp để nhìn rõ năng lực thực sự của model)
    m_log = evaluate_model(y_test_log, y_pred_log)
    
    # 2. Xử lý tập trung: Chặn trần VÀ chặn sàn dự đoán trước khi giải logarit
    # Giới hạn trần ở mức 21 (khoảng 1.3 tỷ USD) để an toàn vượt qua hàm expm1
    # Giới hạn sàn ở 0 để tránh trường hợp expm1 trả về giá nhà âm 
    y_pred_log_safe = np.clip(y_pred_log, a_min=0, a_max=21)
    
    # 3. USD-space — dùng np.expm1 an toàn với dữ liệu đã được clip
    y_pred_usd = np.expm1(np.asarray(y_pred_log_safe, dtype=float))
    m_usd = evaluate_model(y_test_usd, y_pred_usd)
    
    return {
        'Model'      : model_name,
        'MAE (log)'  : m_log['mae'],
        'RMSE (log)' : m_log['rmse'],
        'R² (log)'   : m_log['r2'],
        'MAE ($)'    : m_usd['mae'],
        'RMSE ($)'   : m_usd['rmse'],
        'R² ($)'     : m_usd['r2'],
    }
results = []
predictions = {}

def log_model_progress(step, name, r, extra_info=None):
    """Hàm in kết quả mô hình theo format chuẩn hóa, đẹp mắt."""
    # Định dạng số bước dạng [01/18] để luôn đều nhau
    header = f"[{step:02d}/18] {name}"
    print(f"📌 {header:<45}")
    
    # Nếu có thông tin bổ sung (như lambda hoặc số lượng biến) thì in ra
    if extra_info:
        print(f"   🔹 {extra_info}")
        
    # Căn lề cố định cho các chỉ số để thẳng hàng theo cột
    rmse_str = f"{r['RMSE (log)']:.4f}"
    r2_str = f"{r['R² (log)']:.4f}"
    mae_str = f"${r['MAE ($)']:,.0f}"
    
    print(f"   Metrics ➔  RMSE: {rmse_str:<8} │  R²: {r2_str:<8} │  MAE: {mae_str}")
    print("╌" * 65)

In [ ]:
print('─' * 45)
print('CHUẨN BỊ DỮ LIỆU & SETUP SUBSAMPLE') 
print('─' * 45)

# Đảm bảo y_train là array để không bị lỗi index
y_train_arr = np.asarray(y_train)

# Giữ nguyên y_test gốc và tạo bản log1p
y_test_usd = y_test_raw.copy()          
y_test_log = np.log1p(y_test_raw)       

# Setup index subsample dùng chung cho KRR CV (tránh O(n³))
N_MAX_KRR = 800
if len(y_train) > N_MAX_KRR:
    rng = np.random.default_rng(42)
    idx_cv = rng.choice(len(y_train), N_MAX_KRR, replace=False)
else:
    idx_cv = np.arange(len(y_train))

# =====================================================================
# HÀM HỖ TRỢ TẠO PIPELINE (Dùng cho OLS, Ridge, Lasso)
# =====================================================================


def build_poly_pipeline(estimator):
    return Pipeline([
        ('poly_gen', PolynomialFeatureGenerator(degree=2, top_k=15, use_correlation=True, verbose=False)),
        ('scaler', StandardScaler().set_output(transform="pandas")),
        ('model', estimator)
    ])

def build_inter_pipeline(estimator):
    return Pipeline([
        ('inter_gen', InteractionFeatureGenerator(degree=2, top_k=10, use_correlation=True, verbose=False)),
        ('scaler', StandardScaler().set_output(transform="pandas")),
        ('model', estimator)
    ])

In [ ]:
# =====================================================================
# MÔ HÌNH 1 -> 3: OLS Basic (Dùng bộ dữ liệu đã chuẩn bị ở bước trước)
# =====================================================================
# [1/18] OLS Basic
# [ĐÃ FIX 3]: Thay y_train bằng y_train_arr cho đồng nhất
m1 = OLSBasic(verbose=False).fit(X_train, y_train_arr)
predictions['OLS Basic'] = m1.predict(X_test)
results.append(eval_on_test('OLS Basic', predictions['OLS Basic']))
log_model_progress(1, 'OLS Basic', results[-1])

# [2/18] OLS Basic + Polynomial
m2 = OLSBasic(verbose=False).fit(X_train_poly, y_train_arr)
# [ĐÃ SỬA]: Loại bỏ hoàn toàn np.clip để RMSE phản ánh đúng hiện tượng overfitting/bùng nổ giá trị rác
predictions['OLS Basic + Polynomial'] = m2.predict(X_test_poly)
results.append(eval_on_test('OLS Basic + Polynomial', predictions['OLS Basic + Polynomial']))
log_model_progress(2, 'OLS Basic + Polynomial', results[-1])

# [3/18] OLS Basic + Interaction
m3 = OLSBasic(verbose=False).fit(X_train_inter, y_train_arr)
# [ĐÃ SỬA]: Loại bỏ hoàn toàn np.clip để đảm bảo tính so sánh công bằng giữa các mô hình
predictions['OLS Basic + Interaction'] = m3.predict(X_test_inter)
results.append(eval_on_test('OLS Basic + Interaction', predictions['OLS Basic + Interaction']))
log_model_progress(3, 'OLS Basic + Interaction', results[-1])

In [ ]:
# =====================================================================
# MÔ HÌNH 4 -> 6: OLS Feature Selection
# =====================================================================
# [4/18] OLS Feature Selection
m4 = OLSFeatureSelector(method='both', alpha=0.05, vif_threshold=10.0, verbose=False).fit(X_train, y_train_arr)
predictions['OLS FS'] = m4.predict(X_test)
results.append(eval_on_test('OLS FS', predictions['OLS FS']))
log_model_progress(4, 'OLS Feature Selection', results[-1], f"Biến giữ lại: {len(m4.selected_features_)} / {X_train.shape[1]}")

# [5/18] OLS FS + Polynomial
m5 = OLSFeatureSelector(method='both', alpha=0.05, vif_threshold=10.0, verbose=False).fit(X_train_poly, y_train_arr)
predictions['OLS FS + Polynomial'] = m5.predict(X_test_poly)
results.append(eval_on_test('OLS FS + Polynomial', predictions['OLS FS + Polynomial']))
log_model_progress(5, 'OLS FS + Polynomial', results[-1], f"Biến giữ lại: {len(m5.selected_features_)} / {X_train_poly.shape[1]}")

# [6/18] OLS FS + Interaction
m6 = OLSFeatureSelector(method='both', alpha=0.05, vif_threshold=10.0, verbose=False).fit(X_train_inter, y_train_arr)
predictions['OLS FS + Interaction'] = m6.predict(X_test_inter)
results.append(eval_on_test('OLS FS + Interaction', predictions['OLS FS + Interaction']))
log_model_progress(6, 'OLS FS + Interaction', results[-1], f"Biến giữ lại: {len(m6.selected_features_)} / {X_train_inter.shape[1]}")

In [ ]:
# =====================================================================
# MÔ HÌNH 7 -> 9: Ridge CV
# =====================================================================
# [7/18] Ridge CV
m7 = RidgeCV(lambdas=np.logspace(-4, 4, 80), k_folds=5, verbose=False).fit(X_train, y_train_arr)
predictions['Ridge'] = m7.predict(X_test)
results.append(eval_on_test('Ridge', predictions['Ridge']))
log_model_progress(7, 'Ridge CV', results[-1], f"λ tối ưu: {m7.best_lambda_:.6f}")

# [8/18] Ridge CV + Polynomial
m8 = RidgeCV(lambdas=np.logspace(-4, 4, 80), k_folds=5, verbose=False).fit(X_train_poly, y_train_arr)
predictions['Ridge + Polynomial'] = m8.predict(X_test_poly)
results.append(eval_on_test('Ridge + Polynomial', predictions['Ridge + Polynomial']))
log_model_progress(8, 'Ridge CV + Polynomial', results[-1], f"λ tối ưu: {m8.best_lambda_:.6f}")

# [9/18] Ridge CV + Interaction
m9 = RidgeCV(lambdas=np.logspace(-4, 4, 80), k_folds=5, verbose=False).fit(X_train_inter, y_train_arr)
predictions['Ridge + Interaction'] = m9.predict(X_test_inter)
results.append(eval_on_test('Ridge + Interaction', predictions['Ridge + Interaction']))
log_model_progress(9, 'Ridge CV + Interaction', results[-1], f"λ tối ưu: {m9.best_lambda_:.6f}")

In [ ]:
# =====================================================================
# MÔ HÌNH 10 -> 12: Lasso CV
# =====================================================================
# [10/18] Lasso CV
m10 = LassoCV(lambdas=np.logspace(-6, 0, 50), k_folds=5, verbose=False).fit(X_train, y_train_arr)
predictions['Lasso'] = m10.predict(X_test)
results.append(eval_on_test('Lasso', predictions['Lasso']))
log_model_progress(10, 'Lasso CV', results[-1], f"λ tối ưu: {m10.best_lambda_:.6f}")

# [11/18] Lasso CV + Polynomial
m11 = LassoCV(lambdas=np.logspace(-6, 0, 50), k_folds=5, verbose=False).fit(X_train_poly, y_train_arr)
predictions['Lasso + Polynomial'] = m11.predict(X_test_poly)
results.append(eval_on_test('Lasso + Polynomial', predictions['Lasso + Polynomial']))
log_model_progress(11, 'Lasso CV + Polynomial', results[-1], f"λ tối ưu: {m11.best_lambda_:.6f}")

# [12/18] Lasso CV + Interaction
m12 = LassoCV(lambdas=np.logspace(-6, 0, 50), k_folds=5, verbose=False).fit(X_train_inter, y_train_arr)
predictions['Lasso + Interaction'] = m12.predict(X_test_inter)
results.append(eval_on_test('Lasso + Interaction', predictions['Lasso + Interaction']))
log_model_progress(12, 'Lasso CV + Interaction', results[-1], f"λ tối ưu: {m12.best_lambda_:.6f}")

In [ ]:
# =====================================================================
# MÔ HÌNH 13 -> 15: Kernel Ridge (RBF)
# =====================================================================
n_train = len(y_train_arr)

# Lấy index ngẫu nhiên để tính toán Median Heuristic VÀ chạy CV (Tránh O(n^3))
_sample_idx = np.random.default_rng(42).choice(n_train, min(500, n_train), replace=False)
y_train_sample = y_train_arr[_sample_idx]

# BƯỚC 1: Dùng Median Heuristic để xác định "vùng trọng tâm" cho Length Scale
# Cho OLS Basic
_X_sample   = np.asarray(X_train)[_sample_idx]
_dists_sq   = pairwise_distances(_X_sample, metric='sqeuclidean')
ls_base     = float(np.sqrt(np.median(_dists_sq[_dists_sq > 0])))

# Cho Polynomial
_X_poly_sample = np.asarray(X_train_poly)[_sample_idx]
_dists_sq_poly = pairwise_distances(_X_poly_sample, metric='sqeuclidean')
ls_poly        = float(np.sqrt(np.median(_dists_sq_poly[_dists_sq_poly > 0])))

# Cho Interaction
_X_inter_sample = np.asarray(X_train_inter)[_sample_idx]
_dists_sq_inter = pairwise_distances(_X_inter_sample, metric='sqeuclidean')
ls_inter        = float(np.sqrt(np.median(_dists_sq_inter[_dists_sq_inter > 0])))

# BƯỚC 2: Định nghĩa Grid cho Lambda và chạy CV Search
lam_grid_list = np.logspace(-4, 2, 7).tolist()

# ─────────────────────────────────────────────────────────────────────
# [13/18] Kernel Ridge (RBF) Cơ bản
# ─────────────────────────────────────────────────────────────────────
ls_grid_13 = [ls_base * 0.25, ls_base * 0.5, ls_base, ls_base * 2.0, ls_base * 4.0]

# Chạy tìm kiếm CV thông qua Class Method (CHỈ DÙNG SUBSAMPLE)
cv_res_13 = KernelRidgeRegressor.cv_search(
    X=_X_sample, y=y_train_sample,
    lam_grid=lam_grid_list, ls_grid=ls_grid_13,
    k=5, kernel='rbf', verbose=False
)

# Khởi tạo mô hình với tham số tối ưu và Fit (TRÊN TOÀN BỘ DATA)
m13 = KernelRidgeRegressor(
    kernel='rbf', 
    lam=cv_res_13['best_lam'], 
    length_scale=cv_res_13['best_length_scale']
).fit(X_train, y_train_arr)

predictions['KRR RBF'] = m13.predict(X_test)
results.append(eval_on_test('KRR RBF', predictions['KRR RBF']))
log_model_progress(13, 'Kernel Ridge (RBF)', results[-1], f"λ={m13.lam:.4f}, ls={m13.length_scale:.2f}")

# ─────────────────────────────────────────────────────────────────────
# [14/18] Kernel Ridge (RBF) + Polynomial
# ─────────────────────────────────────────────────────────────────────
ls_grid_14 = [ls_poly * 0.25, ls_poly * 0.5, ls_poly, ls_poly * 2.0, ls_poly * 4.0]

# Chạy tìm kiếm CV (CHỈ DÙNG SUBSAMPLE)
cv_res_14 = KernelRidgeRegressor.cv_search(
    X=_X_poly_sample, y=y_train_sample,
    lam_grid=lam_grid_list, ls_grid=ls_grid_14,
    k=5, kernel='rbf', verbose=False
)

# Khởi tạo mô hình và Fit (TRÊN TOÀN BỘ DATA)
m14 = KernelRidgeRegressor(
    kernel='rbf', 
    lam=cv_res_14['best_lam'], 
    length_scale=cv_res_14['best_length_scale']
).fit(X_train_poly, y_train_arr)

predictions['KRR RBF + Polynomial'] = m14.predict(X_test_poly)
results.append(eval_on_test('KRR RBF + Polynomial', predictions['KRR RBF + Polynomial']))
log_model_progress(14, 'Kernel Ridge (RBF) + Polynomial', results[-1], f"λ={m14.lam:.4f}, ls={m14.length_scale:.2f}")

# ─────────────────────────────────────────────────────────────────────
# [15/18] Kernel Ridge (RBF) + Interaction
# ─────────────────────────────────────────────────────────────────────
ls_grid_15 = [ls_inter * 0.25, ls_inter * 0.5, ls_inter, ls_inter * 2.0, ls_inter * 4.0]

# Chạy tìm kiếm CV (CHỈ DÙNG SUBSAMPLE)
cv_res_15 = KernelRidgeRegressor.cv_search(
    X=_X_inter_sample, y=y_train_sample,
    lam_grid=lam_grid_list, ls_grid=ls_grid_15,
    k=5, kernel='rbf', verbose=False
)

# Khởi tạo mô hình và Fit (TRÊN TOÀN BỘ DATA)
m15 = KernelRidgeRegressor(
    kernel='rbf', 
    lam=cv_res_15['best_lam'], 
    length_scale=cv_res_15['best_length_scale']
).fit(X_train_inter, y_train_arr)

predictions['KRR RBF + Interaction'] = m15.predict(X_test_inter)
results.append(eval_on_test('KRR RBF + Interaction', predictions['KRR RBF + Interaction']))
log_model_progress(15, 'Kernel Ridge (RBF) + Interaction', results[-1], f"λ={m15.lam:.4f}, ls={m15.length_scale:.2f}")

In [ ]:
# =====================================================================
# MÔ HÌNH 16 -> 18: Bayesian Linear Regression
# =====================================================================
alpha_grid = np.logspace(-3, 3, 7) 

# [16/18] Bayesian Linear
X_tr_16, X_val_16, y_tr_16, y_val_16 = train_test_split(X_train, y_train_arr, test_size=0.2, random_state=42)
best_val_rmse, best_alpha = float('inf'), 1.0

for alpha_try in alpha_grid:
    b_temp = BayesianLinearRegressor(alpha=alpha_try).fit(X_tr_16, y_tr_16)
    y_val_p = b_temp.predict(X_val_16, return_std=False)  
    rmse = evaluate_model(y_val_16, y_val_p)['rmse']
    if rmse < best_val_rmse:
        best_val_rmse, best_alpha = rmse, alpha_try

best_bayes = BayesianLinearRegressor(alpha=best_alpha).fit(X_train, y_train_arr)
predictions['Bayesian'] = best_bayes.predict(X_test, return_std=False)
results.append(eval_on_test('Bayesian', predictions['Bayesian']))
log_model_progress(16, 'Bayesian Linear', results[-1], f"α tối ưu={best_alpha:.3f}")

# [17/18] Bayesian Linear + Polynomial
# LƯU Ý: X_train_poly đã được tạo đặc trưng và scale trên toàn bộ X_train 
# trước khi train_test_split. Do đó, tập Validation dưới đây sẽ bị "rò rỉ" 
# (leakage) thông tin nhẹ và có thể trả về RMSE hơi lạc quan.
# Tuy nhiên, kết quả Test cuối cùng vẫn khách quan do X_test hoàn toàn cách ly.
X_tr_17, X_val_17, y_tr_17, y_val_17 = train_test_split(X_train_poly, y_train_arr, test_size=0.2, random_state=42)
best_val_rmse_poly, best_alpha_poly = float('inf'), 1.0

for alpha_try in alpha_grid:
    b_temp = BayesianLinearRegressor(alpha=alpha_try).fit(X_tr_17, y_tr_17)
    y_val_p = b_temp.predict(X_val_17, return_std=False)  
    rmse = evaluate_model(y_val_17, y_val_p)['rmse']
    if rmse < best_val_rmse_poly:
        best_val_rmse_poly, best_alpha_poly = rmse, alpha_try

best_bayes_poly = BayesianLinearRegressor(alpha=best_alpha_poly).fit(X_train_poly, y_train_arr)
predictions['Bayesian + Polynomial'] = best_bayes_poly.predict(X_test_poly, return_std=False)
results.append(eval_on_test('Bayesian + Polynomial', predictions['Bayesian + Polynomial']))
log_model_progress(17, 'Bayesian Linear + Polynomial', results[-1], f"α tối ưu={best_alpha_poly:.3f}")

# [18/18] Bayesian Linear + Interaction
X_tr_18, X_val_18, y_tr_18, y_val_18 = train_test_split(X_train_inter, y_train_arr, test_size=0.2, random_state=42)
best_val_rmse_inter, best_alpha_inter = float('inf'), 1.0

for alpha_try in alpha_grid:
    b_temp = BayesianLinearRegressor(alpha=alpha_try).fit(X_tr_18, y_tr_18)
    y_val_p = b_temp.predict(X_val_18, return_std=False)  
    rmse = evaluate_model(y_val_18, y_val_p)['rmse']
    if rmse < best_val_rmse_inter:
        best_val_rmse_inter, best_alpha_inter = rmse, alpha_try

best_bayes_inter = BayesianLinearRegressor(alpha=best_alpha_inter).fit(X_train_inter, y_train_arr)
predictions['Bayesian + Interaction'] = best_bayes_inter.predict(X_test_inter, return_std=False)
results.append(eval_on_test('Bayesian + Interaction', predictions['Bayesian + Interaction']))
log_model_progress(18, 'Bayesian Linear + Interaction', results[-1], f"α tối ưu={best_alpha_inter:.3f}")

print('\n✅ Hoàn thành huấn luyện 18 mô hình!')

## 4.4. Bảng So Sánh Tổng Hợp

In [ ]:
df_results = pd.DataFrame(results)

df_results['Rank'] = df_results['RMSE (log)'].rank(method='min').astype(int)

def assign_group(model_name):
    base = str(model_name).split(' +')[0].strip()
    if 'KRR' in base or 'Kernel' in base: return 'KRR'
    elif 'Bayesian' in base:              return 'Bayesian'
    elif 'Lasso' in base:                 return 'Lasso'
    elif 'Ridge' in base:                 return 'Ridge'
    elif 'FS' in base or 'Feature Selection' in base: return 'OLS FS'
    else:                                 return 'OLS Basic'

df_results['Nhóm'] = df_results['Model'].apply(assign_group)
df_results = df_results.sort_values('RMSE (log)').reset_index(drop=True)

# Format hiển thị
df_disp = df_results.copy()
df_disp['MAE (log)']  = df_disp['MAE (log)'].map('{:.4f}'.format)
df_disp['RMSE (log)'] = df_disp['RMSE (log)'].map('{:.4f}'.format)
df_disp['R² (log)']   = df_disp['R² (log)'].map('{:.4f}'.format)
df_disp['MAE ($)']    = df_disp['MAE ($)'].map('${:,.0f}'.format)
df_disp['RMSE ($)']   = df_disp['RMSE ($)'].map('${:,.0f}'.format)
df_disp['R² ($)']     = df_disp['R² ($)'].map('{:.4f}'.format)

width = 115
print('\n' + '=' * width)
print('BẢNG SO SÁNH 18 MÔ HÌNH - TEST SET'.center(width))
print('=' * width)
print(df_disp[['Rank', 'Nhóm', 'Model', 'MAE (log)', 'RMSE (log)', 'R² (log)',
               'MAE ($)', 'RMSE ($)', 'R² ($)']].to_string(index=False))
print('=' * width)
print('Ghi chú: log = không gian log1p(SalePrice) | $ = không gian giá gốc (USD)')


## 4.5. Biểu Đồ So Sánh

In [ ]:
# ── 1. LỌC DỮ LIỆU ĐỘNG (BỎ CÁC MÔ HÌNH LỆCH CHUẨN) ───────────────────────────
# Giữ lại các mô hình có MAE (log) <= 5 (loại bỏ KRR bị lỗi)
df_filtered = df_results[df_results['MAE (log)'] <= 5].copy()

# ── 2. CẤU HÌNH MÀU SẮC ────────────────────────────────────────────────────────
GROUP_COLORS = {
    'OLS Basic' : '#2196F3',
    'OLS FS'    : '#00BCD4',
    'Ridge'     : '#4CAF50',
    'Lasso'     : '#8BC34A',
    'KRR'       : '#9C27B0', # Vẫn cứ để màu KRR phòng trường hợp sau này chạy tham số khác ngon hơn
    'Bayesian'  : '#E91E63',
}
BEST_COLOR = '#FF6B35'

def make_colors(vals, groups, higher_is_better=False):
    best_idx = vals.argmax() if higher_is_better else vals.argmin()
    colors = [BEST_COLOR if i == best_idx else GROUP_COLORS.get(g, '#607D8B') 
              for i, g in enumerate(groups)]
    return colors, best_idx

# ── 3. VẼ BIỂU ĐỒ VỚI DỮ LIỆU ĐÃ LỌC ───────────────────────────────────────────
# Đổi biến thành df_filtered nhé
models = df_filtered['Model'].tolist()
groups = df_filtered['Nhóm'].tolist()
x      = np.arange(len(models))
bar_w  = 0.65

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('So Sánh Các Mô Hình Tốt Trên Test Set (Log-scale)', fontsize=16, fontweight='bold', y=1.05)

for ax, (col, ylabel, higher) in zip(axes, [
    ('MAE (log)',  'MAE',  False),
    ('RMSE (log)', 'RMSE', False),
    ('R² (log)',   'R²',   True),
]):
    # Lấy dữ liệu từ df_filtered
    vals = df_filtered[col].values
    colors, best_idx = make_colors(vals, groups, higher)
    bars = ax.bar(x, vals, width=bar_w, color=colors, edgecolor='white', linewidth=0.8, zorder=3)

    ax.autoscale(enable=True, axis='y', tight=False)
    y_min, y_max = ax.get_ylim()
    offset = (y_max - y_min) * 0.02

    for bar, val in zip(bars, vals):
        y_pos    = val + offset if val >= 0 else val - offset
        va_align = 'bottom'     if val >= 0 else 'top'
        ax.text(bar.get_x() + bar.get_width() / 2, y_pos,
                f'{val:.4f}', ha='center', va=va_align,
                fontsize=7.5, fontweight='bold', rotation=45)

    best_y_pos = (vals[best_idx] + offset * 3 
                  if vals[best_idx] >= 0 else vals[best_idx] - offset * 3)
    ax.text(x[best_idx], best_y_pos, '★ BEST', 
            ha='center', fontsize=8.5, color=BEST_COLOR, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'{ylabel} (log-scale)\n{"Cao hơn = tốt hơn" if higher else "Thấp hơn = tốt hơn"}', 
                 fontsize=11, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.yaxis.grid(True, alpha=0.35, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

# Tạo legend dựa trên các nhóm có thực trong df_filtered
active_groups = df_filtered['Nhóm'].unique()
legend_patches = [Patch(color=GROUP_COLORS[g], label=g) for g in active_groups if g in GROUP_COLORS]
legend_patches.append(Patch(color=BEST_COLOR, label='Best overall'))

fig.legend(handles=legend_patches, loc='lower center', ncol=len(active_groups)+1, 
           bbox_to_anchor=(0.5, -0.15), fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.show()

## 4.6. Kết Luận & Chọn Mô Hình Tốt Nhất

In [ ]:
# 1. Dùng df_filtered (đã lọc các mô hình lỗi) và reset lại index
df_print = df_filtered.copy().reset_index(drop=True)

# 2. Sắp xếp theo RMSE (log) từ thấp đến cao
df_print = df_print.sort_values(by='RMSE (log)', ascending=True).reset_index(drop=True)

# ĐÃ XÓA: 3 dòng random seed/uniform ghi đè MAE($) và RMSE($)
# Giữ nguyên cột MAE ($) và RMSE ($) thực từ df_filtered

# ----------------------------------------------------------------
best  = df_print.iloc[0]
worst = df_print.iloc[-1]
rmse_improv = (worst['RMSE (log)'] - best['RMSE (log)']) / worst['RMSE (log)'] * 100

# 3. Xếp hạng động
total_models = len(df_print)
ranks = ['1', '2', '3'] + [f' {i:>2}' for i in range(4, total_models + 1)]
width = 115

print('\n' + '=' * width)
print(f'BẢNG XẾP HẠNG {total_models} MÔ HÌNH THEO RMSE (LOG-SPACE)'.center(width))
print('=' * width)

for idx, row in df_print.iterrows():
    medal = ranks[idx]
    model_str = row['Model'][:35]
    print(f"  {medal}  [{row['Nhóm']:<10}] {model_str:<35} "
          f"RMSE(log)={row['RMSE (log)']:.4f} | R²={row['R² (log)']:.4f} | MAE($)=${row['MAE ($)']:>9,.0f}")

print('=' * width)

model_display = best['Model'][:55] + '...' if len(best['Model']) > 55 else best['Model']
print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║                   MÔ HÌNH ĐƯỢC CHỌN                                  ║
║  ► {model_display:<66}                                               ║
║                                                                      ║
║  Kết quả TEST SET:                                                   ║
║    MAE  (log) = {best['MAE (log)']:<12.4f}                           ║
║    RMSE (log) = {best['RMSE (log)']:<12.4f}                          ║
║    R²   (log) = {best['R² (log)']:<12.4f}                            ║
║    MAE  ($)   = ${best['MAE ($)']:>10,.0f}                           ║
║    RMSE ($)   = ${best['RMSE ($)']:>10,.0f}                          ║
║                                                                      ║
║  Cải thiện RMSE so với mô hình yếu nhất: {rmse_improv:>5.1f}%        ║
╚══════════════════════════════════════════════════════════════════════╝
""")

#### Nhận xét:

##### 1. Lựa chọn mô hình OLS FS (Ordinary Least Squares với Feature Selection)

Mô hình **OLS FS** được lựa chọn làm mô hình cuối cùng nhờ đạt hiệu năng dự đoán tốt nhất trên tập kiểm tra, thể hiện qua giá trị **RMSE** và **MAE** thấp cùng hệ số **R²** cao. Kết quả này cho thấy mô hình không chỉ học tốt trên dữ liệu huấn luyện mà còn có khả năng tổng quát hóa hiệu quả đối với các dữ liệu chưa từng xuất hiện.

Việc kết hợp lựa chọn đặc trưng dựa trên **VIF** và **p-value** đã giúp loại bỏ các biến dư thừa, giảm hiện tượng đa cộng tuyến và giữ lại những đặc trưng có ý nghĩa thống kê quan trọng. Nhờ đó, mô hình đạt được sự cân bằng giữa độ chính xác, tính ổn định và khả năng tổng quát hóa.

Bên cạnh hiệu năng dự báo cao, OLS FS còn nổi bật ở khả năng diễn giải. Các hệ số hồi quy có thể được phân tích trực tiếp để đánh giá mức độ ảnh hưởng của từng đặc trưng đến giá bán nhà, giúp kết quả mô hình minh bạch, dễ giải thích và thuận lợi cho việc ứng dụng trong thực tế.

##### 2. Nguyên nhân mô hình Kernel Ridge Regression (KRR) cho kết quả kém

KRR chịu ảnh hưởng bởi hiện tượng bùng nổ số chiều và đặc tính ma trận thưa của dữ liệu sau One-Hot Encoding. Với hơn 150 biến giả, phần lớn mang giá trị 0, khoảng cách giữa các mẫu trở nên kém ý nghĩa, làm giảm hiệu quả của kernel RBF trong việc đo lường độ tương đồng. Đồng thời, việc sử dụng toàn bộ đặc trưng khiến mô hình dễ bị tác động bởi nhiễu và biến dư thừa, dẫn đến khả năng tổng quát hóa thấp hơn so với OLS FS trên tập kiểm tra.